# ForgeSight — Kaggle (T4)

Thin driver: logic lives in the `forgesight` repo, not in cells.

**Prereqs (do once):**
1. Attach the private dataset **`forgesight-data`** (mounts at `/kaggle/input/datasets/medhulkhandelwal/forgesight-data/`: `train/val/test.jsonl` + `images/`).
2. Add a GitHub PAT to **Add-ons → Secrets** as `GH_PAT` (repo read scope).
3. Accelerator = **GPU T4** (single GPU is enough — 2B/4-bit fits one T4; multi-GPU is not used), Internet = **On**.

Cells: clone → install (incl. bitsandbytes) → import/GPU/data → 4-bit model load (step-9 gate) → **full SFT run** (step 11) → loss curves → val generation peek.

Run cells top to bottom. Step 5 (full training) takes ~1–2h."

In [ ]:
# 1. SINGLE GPU + clone the private repo via PAT
# CUDA_VISIBLE_DEVICES must be set BEFORE torch initializes (cell 3). With 2 GPUs
# visible, HF Trainer wraps the model in nn.DataParallel, which splits the batch
# across GPUs — Qwen2-VL's variable-length image tokens can't be split (vision RoPE
# size mismatch). One GPU also avoids the multi-GPU generate() corruption. 2B/4-bit
# fits one T4 easily.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from kaggle_secrets import UserSecretsClient
pat = UserSecretsClient().get_secret("GH_PAT")
!rm -rf forgesight
!git clone -q https://{pat}@github.com/medhulk8/forgesight.git
%cd forgesight
!git log --oneline -1

In [ ]:
# 2. Install the pinned stack (NOT torch/bitsandbytes — use Kaggle's CUDA builds)
# No `pip install -e .` — cell 3 puts src/ on sys.path (the repo dir shadows the
# package name, so an editable install is unreliable here anyway).
!pip -q install -r requirements-kaggle.txt

In [ ]:
# 3. Sanity: import + data mount + GPU
# The repo dir is named 'forgesight' and sits on sys.path, so a bare `import
# forgesight` grabs it as an EMPTY namespace package, shadowing the real src-layout
# package. Force src/ to the front so the real package wins.
import sys
for _m in [x for x in list(sys.modules) if x == "forgesight" or x.startswith("forgesight.")]:
    del sys.modules[_m]
sys.path.insert(0, "src")
import os, torch, forgesight
from forgesight import schema, coords, collator, model
print("forgesight from:", forgesight.__file__)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| gpus", torch.cuda.device_count())
# Kaggle mounts a dataset at /kaggle/input/datasets/<owner>/<slug>/ (NOT /kaggle/input/<slug>/)
DATA_ROOT = "/kaggle/input/datasets/medhulkhandelwal/forgesight-data"
if not os.path.isdir(DATA_ROOT):                      # fall back if mount layout differs
    hits = [r for r, _, f in os.walk("/kaggle/input") if "train.jsonl" in f]
    DATA_ROOT = hits[0] if hits else DATA_ROOT
print("DATA_ROOT:", DATA_ROOT)
print("data files:", os.listdir(DATA_ROOT))

In [ ]:
# 4. Step 9 gate: 4-bit Qwen2-VL-2B loads on one T4; LoRA params only; one forward pass
from forgesight import model as M
net = M.load_model_for_training(use_4bit=True, attn="sdpa")  # prints trainable params

In [ ]:
# Step 11: FULL QLoRA SFT run (single GPU, ~1-2h on T4). 2 epochs, val eval every 100 steps.
from forgesight.train_sft import load_config, build_trainer
cfg = load_config("configs/sft.yaml"); cfg["data_root"] = DATA_ROOT
trainer, processor, _ = build_trainer(cfg)          # full dataset (no overfit)
trainer.train()
trainer.save_model("/kaggle/working/adapters/sft")
print("adapter saved -> /kaggle/working/adapters/sft")

In [ ]:
# Train + validation loss curves (required deliverable)
import matplotlib.pyplot as plt
h = trainer.state.log_history
tr = [(x["step"], x["loss"]) for x in h if "loss" in x]
ev = [(x["step"], x["eval_loss"]) for x in h if "eval_loss" in x]
plt.figure(figsize=(7, 4))
if tr: plt.plot(*zip(*tr), label="train", alpha=0.7)
if ev: plt.plot(*zip(*ev), marker="o", label="val")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(True)
plt.title("ForgeSight QLoRA SFT"); plt.savefig("/kaggle/working/loss_curve.png", dpi=120); plt.show()
if ev: print("final val loss:", ev[-1][1])

In [ ]:
# Quick generalization peek: generate on 6 VAL examples (unseen docs)
import torch
from datasets import load_dataset
from qwen_vl_utils import process_vision_info
from forgesight.data import conversation
val = load_dataset("json", data_files={"val": f"{DATA_ROOT}/val.jsonl"}, split="val")
model = trainer.model.eval(); dev = next(model.parameters()).device
for i in range(6):
    rec = val[i]
    msgs = conversation.build_messages(rec, data_root=DATA_ROOT, include_target=False)
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    imgs, _ = process_vision_info(msgs)
    ins = processor(text=[text], images=[imgs], return_tensors="pt").to(dev)
    out = model.generate(**ins, max_new_tokens=128, do_sample=False)
    gen = processor.tokenizer.decode(out[0][ins["input_ids"].shape[1]:], skip_special_tokens=False)
    print(f"[{i}] true={rec.get('tamper_type') or 'clean':11s} -> {gen.strip()[:130]}")